### Allscripts Sunrise (SCM) Drug Era Hydration

In [0]:
%sql
TRUNCATE TABLE _exponent.omop_scm.drug_era;

In [0]:
%sql
WITH ingredient_map AS (
  SELECT DISTINCT ds.drug_concept_id, ds.ingredient_concept_id
  FROM _exponent.omop.drug_strength ds
  JOIN _exponent.omop.concept ic
    ON ds.ingredient_concept_id = ic.concept_id
   AND ic.concept_class_id = 'Ingredient'
   AND ic.vocabulary_id IN ('RxNorm', 'RxNorm Extension')
   AND ic.invalid_reason IS NULL
  UNION
  SELECT DISTINCT ca.descendant_concept_id AS drug_concept_id, ca.ancestor_concept_id AS ingredient_concept_id
  FROM _exponent.omop.concept_ancestor ca
  JOIN _exponent.omop.concept ic
    ON ca.ancestor_concept_id = ic.concept_id
   AND ic.concept_class_id = 'Ingredient'
   AND ic.vocabulary_id IN ('RxNorm', 'RxNorm Extension')
   AND ic.invalid_reason IS NULL
), cteDrugPreTarget AS (
  SELECT
      d.drug_exposure_id,
      d.person_id,
      COALESCE(
        ingredient_map.ingredient_concept_id,
        CASE
          WHEN dc.concept_class_id = 'Ingredient'
           AND dc.vocabulary_id IN ('RxNorm', 'RxNorm Extension')
           AND dc.invalid_reason IS NULL
          THEN dc.concept_id
        END
      ) AS ingredient_concept_id,
      d.drug_exposure_start_date,
      d.days_supply,
      COALESCE(
        d.drug_exposure_end_date,
        CASE WHEN d.days_supply IS NOT NULL AND d.days_supply > 0
          THEN date_add(d.drug_exposure_start_date, CAST(d.days_supply AS INT))
        END,
        date_add(d.drug_exposure_start_date, 1)
      ) AS drug_exposure_end_date
  FROM _exponent.omop_scm.drug_exposure d
  LEFT JOIN ingredient_map
    ON ingredient_map.drug_concept_id = d.drug_concept_id
  LEFT JOIN _exponent.omop.concept dc
    ON dc.concept_id = d.drug_concept_id
  WHERE d.drug_concept_id != 0
    AND (d.days_supply IS NULL OR d.days_supply >= 0)
    AND COALESCE(
      ingredient_map.ingredient_concept_id,
      CASE
        WHEN dc.concept_class_id = 'Ingredient'
         AND dc.vocabulary_id IN ('RxNorm', 'RxNorm Extension')
         AND dc.invalid_reason IS NULL
        THEN dc.concept_id
      END
    ) IS NOT NULL
), cteDrugTarget AS (
  SELECT
      drug_exposure_id,
      person_id,
      ingredient_concept_id,
      drug_exposure_start_date,
      days_supply,
      drug_exposure_end_date,
      datediff(drug_exposure_end_date, drug_exposure_start_date) AS days_of_exposure
  FROM cteDrugPreTarget
), cteEndDates AS (
  SELECT person_id, ingredient_concept_id, date_add(event_date, -30) AS end_date
  FROM (
    SELECT
      person_id,
      ingredient_concept_id,
      event_date,
      event_type,
      MAX(start_ordinal) OVER (
        PARTITION BY person_id, ingredient_concept_id
        ORDER BY event_date, event_type
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
      ) AS start_ordinal,
      ROW_NUMBER() OVER (
        PARTITION BY person_id, ingredient_concept_id
        ORDER BY event_date, event_type
      ) AS overall_ord
    FROM (
      SELECT
        person_id,
        ingredient_concept_id,
        drug_exposure_start_date AS event_date,
        -1 AS event_type,
        ROW_NUMBER() OVER (
          PARTITION BY person_id, ingredient_concept_id
          ORDER BY drug_exposure_start_date
        ) AS start_ordinal
      FROM cteDrugTarget
      UNION ALL
      SELECT
        person_id,
        ingredient_concept_id,
        date_add(drug_exposure_end_date, 30) AS event_date,
        1 AS event_type,
        NULL AS start_ordinal
      FROM cteDrugTarget
    ) rawdata
  ) e
  WHERE (2 * e.start_ordinal) - e.overall_ord = 0
), cteDrugExposureEnds AS (
  SELECT
      dt.person_id,
      dt.ingredient_concept_id AS drug_concept_id,
      dt.drug_exposure_start_date,
      MIN(e.end_date) AS drug_era_end_date,
      dt.days_of_exposure
  FROM cteDrugTarget dt
  JOIN cteEndDates e
    ON dt.person_id = e.person_id
   AND dt.ingredient_concept_id = e.ingredient_concept_id
   AND e.end_date >= dt.drug_exposure_start_date
  GROUP BY dt.drug_exposure_id, dt.person_id, dt.ingredient_concept_id, dt.drug_exposure_start_date, dt.days_of_exposure
)
INSERT INTO _exponent.omop_scm.drug_era (
  person_id,
  drug_concept_id,
  drug_era_start_date,
  drug_era_end_date,
  drug_exposure_count,
  gap_days
)
SELECT
  person_id,
  drug_concept_id,
  MIN(drug_exposure_start_date) AS drug_era_start_date,
  drug_era_end_date,
  COUNT(*) AS drug_exposure_count,
  datediff(drug_era_end_date, MIN(drug_exposure_start_date)) - SUM(days_of_exposure) AS gap_days
FROM cteDrugExposureEnds
GROUP BY person_id, drug_concept_id, drug_era_end_date;